In [3]:
import numpy as np
import pandas as pd
import rasterio as rio
from scipy.ndimage import distance_transform_edt
import random
from matplotlib import pyplot as plt
import seaborn as sns
from tqdm import tqdm
from rasterio.transform import Affine
import xgboost as xgb
from xgboost import XGBClassifier
import os, re
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE
from sklearn.feature_selection import RFECV
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.model_selection import GroupKFold
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix

from scipy.spatial.distance import pdist, squareform
import joblib

In [4]:
names = locals()

In [5]:
region_mask = rio.open('../results/region-masks/region-mask-1km.tif').read(1)

In [ ]:
feature_list_1km = dict({
    'elevation': '../results/features/elevation-1km.tif',
    'slope': '../results/features/slope-1km.tif', 
    'population': '../results/population-rasters/population-2020-1km.tif',
    'accessibility': '../results/features/accessibility-1km.tif',
    'D2water': '../results/features/distance-to-water-1km.tif', 
    'forest': '../results/features/forest/forest-2020-1km.tif',
    'cropland': '../results/features/cropland-2019-1km.tif', 
    'water': '../results/mining-area/global-water-area-1km.tif', 
    'su': '../results/features/glim-rasters/su-1km.tif',
    'vb': '../results/features/glim-rasters/vb-1km.tif',
    'ss': '../results/features/glim-rasters/ss-1km.tif',
    'pb': '../results/features/glim-rasters/pb-1km.tif',
    'sm': '../results/features/glim-rasters/sm-1km.tif', 
    'sc': '../results/features/glim-rasters/sc-1km.tif', 
    'va': '../results/features/glim-rasters/va-1km.tif',
    'mt': '../results/features/glim-rasters/mt-1km.tif',
    'pa': '../results/features/glim-rasters/pa-1km.tif',
    'vi': '../results/features/glim-rasters/vi-1km.tif',
    'wb': '../results/features/glim-rasters/wb-1km.tif',
    'py': '../results/features/glim-rasters/py-1km.tif',
    'pi': '../results/features/glim-rasters/pi-1km.tif',
    'ev': '../results/features/glim-rasters/ev-1km.tif',
    'nd': '../results/features/glim-rasters/nd-1km.tif', 
    'ig': '../results/features/glim-rasters/ig-1km.tif', 
    'PA_Ia': '../results/protected-area-rasters/protected-area-Ia-mask.tif',
    'PA_Ib': '../results/protected-area-rasters/protected-area-Ib-mask.tif',
    'PA_II': '../results/protected-area-rasters/protected-area-II-mask.tif',
    'PA_III': '../results/protected-area-rasters/protected-area-III-mask.tif',
    'PA_IV': '../results/protected-area-rasters/protected-area-IV-mask.tif',
    'PA_V': '../results/protected-area-rasters/protected-area-V-mask.tif',
    'PA_VI': '../results/protected-area-rasters/protected-area-VI-mask.tif',
    'urban': '../results/features/urban-area-2020-1km.tif',
    'bare': '../results/features/bare-land-2020-1km.tif',
    'building_stock': '../results/features/building-mineral-stock-1km.tif',
    'building_stock_around': '../results/features/building-mineral-stock-50km-around-1km.tif',
})

In [ ]:
selected_features = ['elevation', 'slope', 'population', 'accessibility', 'D2water', 'forest', 'cropland', 'water', 'su', 'vb', 'ss', 'pb', 'sm', 'sc', 'va', 'mt', 'pa', 'vi', 'wb', 'py', 'pi', 'ev', 'nd', 'ig', 'PA_Ia', 'PA_Ib', 'PA_II', 'PA_III', 'PA_IV', 'PA_V', 'PA_VI', 'urban', 'bare', 'building_stock']
print(len(selected_features)) 

34


In [14]:
def compile_samples_for_region_by_country(the_region, selected_features, res='10km'):
    region_mask_       = rio.open(f'../results/region-masks/region-mask-{res}.tif')
    # country_raster_    = rio.open(f'../results/country-rasters/country-raster-eck4-{res}.tif')
    country_raster_    = rio.open(f'../results/country-rasters/state-raster-eck4-{res}.tif')
    global_quarry_area = rio.open(f'../results/quarry-area-rasters/global-quarry-area-raster-eck4-{res}.tif').read(1)
    # global_quarry_area = rio.open(f'../results/quarry-area-rasters/global-aggregate-quarry-area-raster-eck4-{res}.tif').read(1)

    region_mask    = region_mask_.read(1)
    country_raster = country_raster_.read(1)
    # global_quarry_area = rio.open(f'../results/quarry-area-rasters/global-quarry-area-raster-eck4-{res}.tif').read(1)
    
    feature_list = names[f'feature_list_{res}']

    range_index = np.where(region_mask == the_region)
    i_1 = np.min(range_index[0])
    i_2 = np.max(range_index[0])
    j_1 = np.min(range_index[1])
    j_2 = np.max(range_index[1])
    a_window = ((i_1, i_2), (j_1, j_2))

    presence_in_region_ = global_quarry_area[i_1:i_2, j_1:j_2]
    region_mask = region_mask[i_1:i_2, j_1:j_2] == the_region
    country_mask = country_raster[i_1:i_2, j_1:j_2]
    presence_in_region = np.where(region_mask & (presence_in_region_ > 0), 1, 0).astype(np.uint8)

    # GLWD = GLWD_.read(1, window=a_window)
    # valid_GLWD_classes = [0] + [i for i in range(8, 34)]

    distance_min, distance_max = 50, 100
    r = int(re.findall(r'\d+', res)[0])
    D_min, D_max = distance_min//r, distance_max//r
    distance_from_quarry = distance_transform_edt(presence_in_region == 0)


    # ----------------------------
    # 4. 初始化伪负样本矩阵
    # ----------------------------
    pseudoabsence_total = np.zeros_like(presence_in_region, dtype=np.uint8)

    country_ids = np.unique(country_mask[region_mask])
    country_ids = country_ids[country_ids > 0]

    for cid in tqdm(country_ids, desc=f'Region {the_region}'):
        country_mask_local = (country_mask == cid)
        presence_in_country = presence_in_region & country_mask_local
        n_presence = np.sum(presence_in_country)
        if n_presence == 0:
            continue  # 该国没有正样本，跳过

        # 候选伪负样本点：满足距离约束 + 在该国范围 + 属于该region
        pseudo_candidates = np.where(
            (distance_from_quarry >= D_min) & 
            (distance_from_quarry <= D_max) &
            (country_mask_local)
        )

        n_candidates = len(pseudo_candidates[0])
        if n_candidates == 0:
            continue

        n_pseudo = n_presence if n_candidates > n_presence else n_candidates
        random.seed(42)
        selected_idx = random.sample(range(n_candidates), n_pseudo)

        pseudoabsence_total[
            pseudo_candidates[0][selected_idx],
            pseudo_candidates[1][selected_idx]
        ] = 1

        print(f"Region {the_region}, Country {cid}: presence={n_presence}, pseudo={n_pseudo}")
    
    # ----------------------------
    # 5. 生成标签与特征
    # ----------------------------
    label_mask = (presence_in_region == 1) | (pseudoabsence_total == 1)
    label_array = np.where(presence_in_region == 1, 1, np.where(pseudoabsence_total == 1, 0, np.nan)).astype(np.float32) 

    feature_values = []
    for feature in selected_features:
        a_path = feature_list[feature]
        src = rio.open(a_path)
        gg = src.read(1, window=a_window)
        hh = gg[label_mask]
        feature_values.append(hh)
    
    # feature_values = [feature[label_mask] for feature in [elevation, slope, population, accessibility, D2water, treecover, cropland, lithology, coarse_extraction, fine_extraction, limestone_extraction]]

    Y_ = label_array[label_mask]
    X_ = np.array(feature_values).T
    valid_mask = ~np.isnan(X_).any(axis=1)
    X = X_[valid_mask]
    Y = Y_[valid_mask]

    country_ids_array = country_mask[label_mask]
    country_ids_for_samples = country_ids_array[valid_mask]

    value_mask_ = np.where(label_mask)
    value_mask = (value_mask_[0][valid_mask], value_mask_[1][valid_mask])
    
    return X, Y, country_ids_for_samples, (i_1, i_2, j_1, j_2), region_mask, value_mask

In [21]:
X_global, Y_global, country_labels = [], [], []
for i in tqdm(range(1, 16)):
    X_region, Y_region, country_labels_region, *_ = compile_samples_for_region_by_country(i, selected_features, res='1km')
    X_global.append(X_region)
    Y_global.append(Y_region)
    country_labels.append(country_labels_region)

  0%|          | 0/15 [00:00<?, ?it/s]

Region 1, Country 1: presence=2888, pseudo=2888
Region 1, Country 41: presence=161, pseudo=161
Region 1, Country 57: presence=1174, pseudo=1174
Region 1, Country 65: presence=316, pseudo=246
Region 1, Country 73: presence=5229, pseudo=191


Region 1, Country 107: presence=273, pseudo=273
Region 1, Country 120: presence=448, pseudo=448
Region 1, Country 162: presence=701, pseudo=701


Region 1: 100%|██████████| 17/17 [00:00<00:00, 22.04it/s]

Region 1, Country 191: presence=126, pseudo=126
Region 1, Country 194: presence=3081, pseudo=3081
Region 1, Country 236: presence=49, pseudo=49


...

Region 15, Country 28: presence=17, pseudo=17
Region 15, Country 29: presence=216, pseudo=216


Region 15, Country 31: presence=2699, pseudo=2699
Region 15, Country 32: presence=1066, pseudo=1066


Region 15: 100%|██████████| 32/32 [00:03<00:00,  9.59it/s]

Region 15, Country 33: presence=644, pseudo=644
Region 15, Country 34: presence=210, pseudo=210



100%|██████████| 15/15 [05:41<00:00, 22.74s/it]


In [22]:
results_all = []

for region_id in range(1, 16):
    X_region, Y_region = X_global[region_id-1], Y_global[region_id-1]
    country_labels_region = country_labels[region_id-1]
    unique_countries = np.unique(country_labels_region)
    n_countries = len(unique_countries)
    # print(n_countries)

    if n_countries == 1:
        print(f"⚠️ Region {region_id}: only one country, skip CV")
        continue
    elif n_countries <= 3:
        print(f"Region {region_id}: using Leave-One-Group-Out CV ({n_countries} countries)")
        cv = LeaveOneGroupOut()
    else:
        n_splits = min(5, n_countries)
        cv = GroupKFold(n_splits=n_splits)
        print(f"Region {region_id}: using GroupKFold with {n_splits} folds ({n_countries} countries)")
    
    region_results = []

    for fold, (train_idx, test_idx) in enumerate(cv.split(X_region, Y_region, groups=country_labels_region), 1):
        train_countries = np.unique(country_labels_region[train_idx])
        test_countries  = np.unique(country_labels_region[test_idx])

        # print(f"  Fold {fold}: train {len(train_countries)} countries, test {len(test_countries)} countries")

        # 取训练 / 测试集
        X_train, X_test = X_region[train_idx], X_region[test_idx]
        Y_train, Y_test = Y_region[train_idx], Y_region[test_idx]

        # 4️⃣ 训练模型
        model = RandomForestClassifier(
            n_estimators=400,
            max_depth=None,
            n_jobs=-1,
            random_state=42
        )
        model.fit(X_train, Y_train)

        # 5️⃣ 测试与评估
        y_pred_prob = model.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(Y_test, y_pred_prob)
        ap = average_precision_score(Y_test, y_pred_prob)

        region_results.append({
            'region': region_id,
            'fold': fold,
            'n_train_countries': len(train_countries),
            'n_test_countries': len(test_countries),
            'auc': auc,
            'ap': ap
        })

        print(f"  Fold {fold}: train {len(train_countries)} countries, test {len(test_countries)} countries: AUC={auc:.3f}, AP={ap:.3f}")

    # 6️⃣ 汇总该 region 的结果
    df_region = pd.DataFrame(region_results)
    print(df_region[['auc','ap']].mean())
    results_all.append(df_region)

# 7️⃣ 汇总所有 region 的结果
df_all = pd.concat(results_all, ignore_index=True)
print("\n===== Overall summary =====")
print(df_all.groupby('region')[['auc','ap']].mean())

Region 1: using GroupKFold with 5 folds (16 countries)
  Fold 1: train 15 countries, test 1 countries: AUC=0.785, AP=0.803
  Fold 2: train 15 countries, test 1 countries: AUC=0.836, AP=0.793
  Fold 3: train 15 countries, test 1 countries: AUC=0.884, AP=0.995
  Fold 4: train 10 countries, test 6 countries: AUC=0.972, AP=0.981
  Fold 5: train 9 countries, test 7 countries: AUC=0.907, AP=0.908
auc    0.876495
ap     0.895993
dtype: float64
Region 2: using GroupKFold with 5 folds (48 countries)
  Fold 1: train 47 countries, test 1 countries: AUC=0.794, AP=0.808
  Fold 2: train 47 countries, test 1 countries: AUC=0.792, AP=0.816
  Fold 3: train 33 countries, test 15 countries: AUC=0.844, AP=0.858
  Fold 4: train 32 countries, test 16 countries: AUC=0.870, AP=0.871
  Fold 5: train 33 countries, test 15 countries: AUC=0.662, AP=0.668
auc    0.792182
ap     0.804056
dtype: float64
Region 3: using GroupKFold with 5 folds (7 countries)
  Fold 1: train 6 countries, test 1 countries: AUC=0.670, AP

C:\Users\Administrator\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


  Fold 4: train 3 countries, test 1 countries: AUC=nan, AP=1.000
auc    0.754425
ap     0.794689
dtype: float64
Region 13: using GroupKFold with 5 folds (10 countries)
  Fold 1: train 9 countries, test 1 countries: AUC=0.773, AP=0.819
  Fold 2: train 9 countries, test 1 countries: AUC=0.896, AP=0.896
  Fold 3: train 7 countries, test 3 countries: AUC=0.801, AP=0.708
  Fold 4: train 8 countries, test 2 countries: AUC=0.865, AP=0.892
  Fold 5: train 7 countries, test 3 countries: AUC=0.882, AP=0.944
auc    0.843317
ap     0.851902
dtype: float64
Region 14: using GroupKFold with 5 folds (64 countries)
  Fold 1: train 53 countries, test 11 countries: AUC=0.956, AP=0.968
  Fold 2: train 53 countries, test 11 countries: AUC=0.878, AP=0.926
  Fold 3: train 50 countries, test 14 countries: AUC=0.942, AP=0.954
  Fold 4: train 51 countries, test 13 countries: AUC=0.882, AP=0.906
  Fold 5: train 49 countries, test 15 countries: AUC=0.848, AP=0.858
auc    0.901226
ap     0.922443
dtype: float64
Re

In [ ]:
models_by_region = {}

for ridx, (Xr, Yr) in enumerate(zip(X_global, Y_global), start=1):
    print(f"Training final model for Region {ridx} ...")
    
    model = XGBClassifier(
        objective='binary:logistic',
        eval_metric='aucpr',
        tree_method='hist',
        n_estimators=800,
        learning_rate=0.05,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=3,
        random_state=42,
        n_jobs=-1
    )
    model.fit(Xr, Yr)
    models_by_region[ridx] = model
    joblib.dump(model, f'../results/trained-models/xgb-region-{ridx}.pkl')  # store the trained RF models

Training final model for Region 1 ...
Training final model for Region 2 ...
Training final model for Region 3 ...
Training final model for Region 4 ...
Training final model for Region 5 ...
Training final model for Region 6 ...
Training final model for Region 7 ...
Training final model for Region 8 ...
Training final model for Region 9 ...
Training final model for Region 10 ...
Training final model for Region 11 ...
Training final model for Region 12 ...
Training final model for Region 13 ...
Training final model for Region 14 ...
Training final model for Region 15 ...
